In [ ]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# CONFIG
CSV_PATH = "/content/final_traffic_weather_merged.csv"
TARGET_HORIZON = 1
TEST_RATIO = 0.2
RANDOM_STATE = 42


# GROUPED BY centerline_id + direction
GROUP_COLS = ["centreline_id", "direction"]

# LOAD DATA
df = pd.read_csv(CSV_PATH)

print("Original columns:")
print(df.columns.tolist())


# CLEAN COLUMN NAMES
df.columns = (
    df.columns
    .str.strip()
    .str.replace(" ", "_", regex=False)
    .str.replace("(", "", regex=False)
    .str.replace(")", "", regex=False)
    .str.replace(".", "", regex=False)
    .str.replace("/", "_", regex=False)
    .str.replace("%", "pct", regex=False)
    .str.replace("°", "deg", regex=False)
)

print("\nCleaned columns:")
print(df.columns.tolist())
print(df["centreline_id"].nunique())
if "centreline_id" in df.columns and "centerline_id" not in df.columns:
    df = df.rename(columns={"centreline_id": "centerline_id"})

# HANDLE COLUMN NAMES
required_cols = [
    "centerline_id",
    "location_name",
    "direction",
    "time_start",
    "volume_15min",
    "avg_speed_kph"
]

for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Required column missing: {col}")

df["time_start"] = pd.to_datetime(df["time_start"], errors="coerce")
df = df.dropna(subset=["time_start"])

for col in ["is_raining", "is_snowing", "is_foggy"]:
    if col in df.columns:
        df[col] = (
            df[col]
            .replace({
                True: 1, False: 0,
                "TRUE": 1, "FALSE": 0,
                "True": 1, "False": 0,
                "yes": 1, "no": 0,
                "Yes": 1, "No": 0
            })
            .fillna(0)
            .astype(int)
        )

numeric_cols_to_convert = [
    "longitude", "latitude", "volume_15min", "avg_speed_kph",
    "Temp_degC", "Precip_Amount_mm", "Wind_Spd_km_h",
    "Visibility_km", "Rel_Hum_pct", "Wind_Sin", "Wind_Cos"
]

for col in numeric_cols_to_convert:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

GROUP_COLS = ["centerline_id", "direction"]

df = df.sort_values(GROUP_COLS + ["time_start"]).reset_index(drop=True)

# BASIC TIME FEATURES
df["hour"] = df["time_start"].dt.hour
df["minute"] = df["time_start"].dt.minute
df["day_of_week"] = df["time_start"].dt.dayofweek
df["month"] = df["time_start"].dt.month
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)


# LAG FEATURES
# lag_1 = 15 mins before, lag_2 = 30 mins before, lag_4 = 1 hour before
lag_steps = [1, 2, 4, 8]

for lag in lag_steps:
    df[f"speed_lag_{lag}"] = (
        df.groupby(GROUP_COLS)["avg_speed_kph"].shift(lag)
    )
    df[f"volume_lag_{lag}"] = (
        df.groupby(GROUP_COLS)["volume_15min"].shift(lag)
    )

# ROLLING FEATURES
rolling_windows = [4, 8]   # 4=past 1 hour, 8=past 2 hours

for window in rolling_windows:
    df[f"speed_roll_mean_{window}"] = (
        df.groupby(GROUP_COLS)["avg_speed_kph"]
        .transform(lambda x: x.shift(1).rolling(window=window, min_periods=1).mean())
    )
    df[f"speed_roll_std_{window}"] = (
        df.groupby(GROUP_COLS)["avg_speed_kph"]
        .transform(lambda x: x.shift(1).rolling(window=window, min_periods=1).std())
    )
    df[f"volume_roll_mean_{window}"] = (
        df.groupby(GROUP_COLS)["volume_15min"]
        .transform(lambda x: x.shift(1).rolling(window=window, min_periods=1).mean())
    )


# TARGET: FUTURE SPEED
# predict avg_speed_kph after 15 mins
df["target_speed"] = (
    df.groupby(GROUP_COLS)["avg_speed_kph"].shift(-TARGET_HORIZON)
)

df = df.dropna(subset=["target_speed"]).copy()


# FEATURE LIST
numeric_features = [
    "longitude",
    "latitude",
    "volume_15min",
    "avg_speed_kph",
    "Temp_degC",
    "Precip_Amount_mm",
    "Wind_Spd_km_h",
    "Visibility_km",
    "Rel_Hum_pct",
    "Wind_Sin",
    "Wind_Cos",
    "is_raining",
    "is_snowing",
    "is_foggy",
    "hour",
    "minute",
    "day_of_week",
    "month",
    "is_weekend",
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
] + [f"speed_lag_{lag}" for lag in lag_steps] \
  + [f"volume_lag_{lag}" for lag in lag_steps] \
  + [f"speed_roll_mean_{w}" for w in rolling_windows] \
  + [f"speed_roll_std_{w}" for w in rolling_windows] \
  + [f"volume_roll_mean_{w}" for w in rolling_windows]

categorical_features = [
    "direction"
]

numeric_features += ["centerline_id"]
categorical_features = [c for c in categorical_features if c in df.columns]

all_features = numeric_features + categorical_features

print("\nNumber of features:", len(all_features))
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

# DROP ROWS WITH TOO MANY MISSING VALUES
# simple baseline
model_df = df[["time_start", "target_speed"] + all_features].copy()



# TIME-BASED TRAIN / TEST SPLIT
model_df = model_df.sort_values("time_start").reset_index(drop=True)

split_idx = int(len(model_df) * (1 - TEST_RATIO))
train_df = model_df.iloc[:split_idx].copy()
test_df = model_df.iloc[split_idx:].copy()

X_train = train_df[all_features]
y_train = train_df["target_speed"]

X_test = test_df[all_features]
y_test = test_df["target_speed"]

print("\nTrain size:", len(train_df))
print("Test size:", len(test_df))
print("Train time range:", train_df["time_start"].min(), "to", train_df["time_start"].max())
print("Test time range:", test_df["time_start"].min(), "to", test_df["time_start"].max())


# PREPROCESSOR
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)


# BASELINE MODEL
baseline_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    min_samples_split=10,
    min_samples_leaf=4,
    n_jobs=-1,
    random_state=RANDOM_STATE
)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", baseline_model)
])


# TRAIN
pipeline.fit(X_train, y_train)


# PREDICT
y_pred = pipeline.predict(X_test)


# EVALUATE
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\n===== Baseline Model Performance =====")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R^2  : {r2:.4f}")

# SAVE PREDICTIONS
result_df = test_df[["time_start"]].copy()
result_df["y_true"] = y_test.values
result_df["y_pred"] = y_pred
result_df.to_csv("baseline_predictions.csv", index=False)
print("\nSaved predictions to baseline_predictions.csv")


# FEATURE IMPORTANCE

ohe = pipeline.named_steps["preprocessor"].named_transformers_["cat"].named_steps["onehot"]
cat_feature_names = ohe.get_feature_names_out(categorical_features)

final_feature_names = numeric_features + list(cat_feature_names)

importances = pipeline.named_steps["model"].feature_importances_
importance_df = pd.DataFrame({
    "feature": final_feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

print("\nTop 20 important features:")
print(importance_df.head(20))

importance_df.to_csv("baseline_feature_importance.csv", index=False)
print("\nSaved feature importance to baseline_feature_importance.csv")

Original columns:
['centreline_id', 'location_name', 'longitude', 'latitude', 'direction', 'time_start', 'time_end', 'volume_15min', 'avg_speed_kph', 'Temp (°C)', 'Precip. Amount (mm)', 'Wind Spd (km/h)', 'Visibility (km)', 'Rel Hum (%)', 'Wind_Sin', 'Wind_Cos', 'is_raining', 'is_snowing', 'is_foggy']

Cleaned columns:
['centreline_id', 'location_name', 'longitude', 'latitude', 'direction', 'time_start', 'time_end', 'volume_15min', 'avg_speed_kph', 'Temp_degC', 'Precip_Amount_mm', 'Wind_Spd_km_h', 'Visibility_km', 'Rel_Hum_pct', 'Wind_Sin', 'Wind_Cos', 'is_raining', 'is_snowing', 'is_foggy']
2946

Number of features: 39
Numeric features: ['longitude', 'latitude', 'volume_15min', 'avg_speed_kph', 'Temp_degC', 'Precip_Amount_mm', 'Wind_Spd_km_h', 'Visibility_km', 'Rel_Hum_pct', 'Wind_Sin', 'Wind_Cos', 'is_raining', 'is_snowing', 'is_foggy', 'hour', 'minute', 'day_of_week', 'month', 'is_weekend', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'speed_lag_1', 'speed_lag_2', 'speed_lag_4', 's

In [ ]:
# ============================================================
# Fair Comparison: XGBoost vs KNN vs LightGBM
# Same basic feature set as RF baseline
# Target: next 15-min avg_speed_kph
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.neighbors import KNeighborsRegressor

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# CONFIG
CSV_PATH = "/content/final_traffic_weather_merged.csv"

TARGET_HORIZON = 1
RANDOM_STATE = 42

# train/vali/test：70 / 10 / 20
TRAIN_RATIO = 0.70
VAL_RATIO = 0.10
TEST_RATIO = 0.20

# KNN is slow for huge set, substract
USE_SUBSAMPLE_FOR_KNN = True
KNN_TRAIN_MAX = 120000
KNN_N_NEIGHBORS = 15

GROUP_COLS = ["centreline_id", "direction"]

# LOAD DATA
df = pd.read_csv(CSV_PATH)

print("Original columns:")
print(df.columns.tolist())

# time
df["time_start"] = pd.to_datetime(df["time_start"], errors="coerce")
df["time_end"] = pd.to_datetime(df["time_end"], errors="coerce")
df = df.dropna(subset=["time_start"]).copy()

# sort
df = df.sort_values(GROUP_COLS + ["time_start"]).reset_index(drop=True)

# boolean
for col in ["is_raining", "is_snowing", "is_foggy"]:
    if col in df.columns:
        df[col] = (
            df[col]
            .replace({True: 1, False: 0, "TRUE": 1, "FALSE": 0, "True": 1, "False": 0})
            .fillna(0)
            .astype(int)
        )

# columns numeric
num_convert_cols = [
    "longitude", "latitude", "volume_15min", "avg_speed_kph",
    "Temp (°C)", "Precip. Amount (mm)", "Wind Spd (km/h)",
    "Visibility (km)", "Rel Hum (%)", "Wind_Sin", "Wind_Cos"
]
for col in num_convert_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print(f"\nTotal raw rows: {len(df)}")


# BASIC TIME FEATURES
df["hour"] = df["time_start"].dt.hour
df["minute"] = df["time_start"].dt.minute
df["day_of_week"] = df["time_start"].dt.dayofweek
df["month"] = df["time_start"].dt.month
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)

# LAG FEATURES
lag_steps = [1, 2, 4, 8]

for lag in lag_steps:
    df[f"speed_lag_{lag}"] = df.groupby(GROUP_COLS)["avg_speed_kph"].shift(lag)
    df[f"volume_lag_{lag}"] = df.groupby(GROUP_COLS)["volume_15min"].shift(lag)

# ROLLING FEATURES
rolling_windows = [4, 8]

for window in rolling_windows:
    df[f"speed_roll_mean_{window}"] = (
        df.groupby(GROUP_COLS)["avg_speed_kph"]
        .transform(lambda x: x.shift(1).rolling(window=window, min_periods=1).mean())
    )
    df[f"speed_roll_std_{window}"] = (
        df.groupby(GROUP_COLS)["avg_speed_kph"]
        .transform(lambda x: x.shift(1).rolling(window=window, min_periods=1).std())
    )
    df[f"volume_roll_mean_{window}"] = (
        df.groupby(GROUP_COLS)["volume_15min"]
        .transform(lambda x: x.shift(1).rolling(window=window, min_periods=1).mean())
    )

# TARGET
# future 15 mins avg_speed_kph
df["target_speed"] = df.groupby(GROUP_COLS)["avg_speed_kph"].shift(-TARGET_HORIZON)

# FEATURE LIST
# all model use same simple feature
numeric_features = [
    "centreline_id",
    "longitude",
    "latitude",
    "volume_15min",
    "avg_speed_kph",
    "Temp (°C)",
    "Precip. Amount (mm)",
    "Wind Spd (km/h)",
    "Visibility (km)",
    "Rel Hum (%)",
    "Wind_Sin",
    "Wind_Cos",
    "is_raining",
    "is_snowing",
    "is_foggy",
    "hour",
    "minute",
    "day_of_week",
    "month",
    "is_weekend",
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
] + [f"speed_lag_{lag}" for lag in lag_steps] \
  + [f"volume_lag_{lag}" for lag in lag_steps] \
  + [f"speed_roll_mean_{w}" for w in rolling_windows] \
  + [f"speed_roll_std_{w}" for w in rolling_windows] \
  + [f"volume_roll_mean_{w}" for w in rolling_windows]

categorical_features = [
    "direction"
]

numeric_features = [c for c in numeric_features if c in df.columns]
categorical_features = [c for c in categorical_features if c in df.columns]
all_features = numeric_features + categorical_features

print("\nNumber of final features:", len(all_features))
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

model_df = df[["time_start", "target_speed"] + all_features].copy()
model_df = model_df.dropna(subset=["target_speed"]).copy()

print(f"\nRows before split: {len(model_df)}")

# TIME-BASED SPLIT
model_df = model_df.sort_values("time_start").reset_index(drop=True)

n = len(model_df)
train_end = int(n * TRAIN_RATIO)
val_end = int(n * (TRAIN_RATIO + VAL_RATIO))

train_df = model_df.iloc[:train_end].copy()
val_df = model_df.iloc[train_end:val_end].copy()
test_df = model_df.iloc[val_end:].copy()

print(f"\nTrain size: {len(train_df)}")
print(f"Val size:   {len(val_df)}")
print(f"Test size:  {len(test_df)}")
print(f"Train time range: {train_df['time_start'].min()} to {train_df['time_start'].max()}")
print(f"Val time range:   {val_df['time_start'].min()} to {val_df['time_start'].max()}")
print(f"Test time range:  {test_df['time_start'].min()} to {test_df['time_start'].max()}")

X_train = train_df[all_features]
y_train = train_df["target_speed"]

X_val = val_df[all_features]
y_val = val_df["target_speed"]

X_test = test_df[all_features]
y_test = test_df["target_speed"]

# PREPROCESSORS
tree_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ]), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features)
    ]
)

knn_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features)
    ]
)

# METRIC FUNCTION
def evaluate_regression(y_true, y_pred, model_name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return {
        "model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }

results = []

# XGBOOST
print("\n================ XGBOOST ================")

X_train_tree = tree_preprocessor.fit_transform(X_train)
X_val_tree = tree_preprocessor.transform(X_val)
X_test_tree = tree_preprocessor.transform(X_test)

xgb_model = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=8,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    tree_method="hist",
    objective="reg:squarederror",
    eval_metric="rmse",
    early_stopping_rounds=30,
    n_jobs=-1,
    random_state=RANDOM_STATE
)

xgb_model.fit(
    X_train_tree, y_train,
    eval_set=[(X_val_tree, y_val)],
    verbose=100
)

xgb_pred = xgb_model.predict(X_test_tree)
results.append(evaluate_regression(y_test, xgb_pred, "XGBoost"))

# LIGHTGBM
print("\n================ LIGHTGBM ================")

lgb_model = LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=8,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="regression",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

lgb_model.fit(
    X_train_tree, y_train,
    eval_set=[(X_val_tree, y_val)],
    eval_metric="rmse",
    callbacks=[]
)

lgb_pred = lgb_model.predict(X_test_tree)
results.append(evaluate_regression(y_test, lgb_pred, "LightGBM"))

# KNN
print("\n================ KNN ================")
X_train_knn = knn_preprocessor.fit_transform(X_train)
X_test_knn = knn_preprocessor.transform(X_test)
y_train_knn = y_train.copy()

if USE_SUBSAMPLE_FOR_KNN and X_train_knn.shape[0] > KNN_TRAIN_MAX:
    rng = np.random.default_rng(RANDOM_STATE)
    idx = rng.choice(X_train_knn.shape[0], size=KNN_TRAIN_MAX, replace=False)
    X_train_knn_used = X_train_knn[idx]
    y_train_knn_used = y_train_knn.iloc[idx]
    print(f"KNN uses subsampled train size: {len(y_train_knn_used)}")
else:
    X_train_knn_used = X_train_knn
    y_train_knn_used = y_train_knn
    print(f"KNN uses full train size: {len(y_train_knn_used)}")

knn_model = KNeighborsRegressor(
    n_neighbors=KNN_N_NEIGHBORS,
    weights="distance",
    metric="minkowski",
    p=2,
    algorithm="brute",
    n_jobs=-1
)

knn_model.fit(X_train_knn_used, y_train_knn_used)
knn_pred = knn_model.predict(X_test_knn)
results.append(evaluate_regression(y_test, knn_pred, "KNN"))

# RESULTS
results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)

print("\n================ FINAL COMPARISON ================")
print(results_df)

results_df.to_csv("model_comparison_same_features.csv", index=False)
print("\nSaved results to model_comparison_same_features.csv")

# FEATURE IMPORTANCE
try:
    feature_names_cat = []
    if len(categorical_features) > 0:
        ohe = tree_preprocessor.named_transformers_["cat"].named_steps["onehot"]
        feature_names_cat = list(ohe.get_feature_names_out(categorical_features))

    final_feature_names = numeric_features + feature_names_cat

    xgb_importance_df = pd.DataFrame({
        "feature": final_feature_names,
        "importance": xgb_model.feature_importances_
    }).sort_values("importance", ascending=False)

    lgb_importance_df = pd.DataFrame({
        "feature": final_feature_names,
        "importance": lgb_model.feature_importances_
    }).sort_values("importance", ascending=False)

    print("\nTop 20 XGBoost features:")
    print(xgb_importance_df.head(20))

    print("\nTop 20 LightGBM features:")
    print(lgb_importance_df.head(20))

    xgb_importance_df.to_csv("xgb_feature_importance_same_features.csv", index=False)
    lgb_importance_df.to_csv("lgb_feature_importance_same_features.csv", index=False)
except Exception as e:
    print("\nFeature importance export skipped:", e)

Original columns:
['centreline_id', 'location_name', 'longitude', 'latitude', 'direction', 'time_start', 'time_end', 'volume_15min', 'avg_speed_kph', 'Temp (°C)', 'Precip. Amount (mm)', 'Wind Spd (km/h)', 'Visibility (km)', 'Rel Hum (%)', 'Wind_Sin', 'Wind_Cos', 'is_raining', 'is_snowing', 'is_foggy']

Total raw rows: 1737739

Number of final features: 39
Numeric features: ['centreline_id', 'longitude', 'latitude', 'volume_15min', 'avg_speed_kph', 'Temp (°C)', 'Precip. Amount (mm)', 'Wind Spd (km/h)', 'Visibility (km)', 'Rel Hum (%)', 'Wind_Sin', 'Wind_Cos', 'is_raining', 'is_snowing', 'is_foggy', 'hour', 'minute', 'day_of_week', 'month', 'is_weekend', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'speed_lag_1', 'speed_lag_2', 'speed_lag_4', 'speed_lag_8', 'volume_lag_1', 'volume_lag_2', 'volume_lag_4', 'volume_lag_8', 'speed_roll_mean_4', 'speed_roll_mean_8', 'speed_roll_std_4', 'speed_roll_std_8', 'volume_roll_mean_4', 'volume_roll_mean_8']
Categorical features: ['direction']

Rows b